<a href="https://www.kaggle.com/code/rentoda/japanese-wiki-table?scriptVersionId=335631281" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [5]:
!pip install romkan2
!pip install namedivider-python
!pip install namedivider-core

In [6]:
QUERY = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX schema: <http://schema.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?person ?personLabel ?personLabelEn ?article ?genderLabel ?birthDate ?deathDate ?description
       ?image
       (GROUP_CONCAT(DISTINCT ?educationLabel; separator="|") AS ?educations)
       (GROUP_CONCAT(DISTINCT ?occupationLabel; separator="|") AS ?occupations)
       (GROUP_CONCAT(DISTINCT ?familyNameLabel; separator="|") AS ?familyNames)
       (GROUP_CONCAT(DISTINCT ?givenNameLabel; separator="|") AS ?givenNames)
       ?height ?weight
       (GROUP_CONCAT(DISTINCT ?causeOfDeathLabel; separator="|") AS ?causesOfDeath)
       (GROUP_CONCAT(DISTINCT ?mannerOfDeathLabel; separator="|") AS ?mannersOfDeath)
       WHERE {
  ?person wdt:P31 wd:Q5 ;
          wdt:P27 wd:Q17 .
  ?article schema:about ?person ;
           schema:inLanguage "ja" ;
           schema:isPartOf <https://ja.wikipedia.org/> .
  OPTIONAL { ?person rdfs:label ?personLabel . FILTER(LANG(?personLabel) = "ja") }
  OPTIONAL { ?person rdfs:label ?personLabelEn . FILTER(LANG(?personLabelEn) = "en") }
  OPTIONAL {
    ?person wdt:P21 ?gender.
    ?gender rdfs:label ?genderLabel . FILTER(LANG(?genderLabel) = "ja")
  }
  OPTIONAL { ?person wdt:P569 ?birthDate. }
  OPTIONAL { ?person wdt:P570 ?deathDate. }
  OPTIONAL {
    ?person schema:description ?description.
    FILTER(LANG(?description) = "ja")
  }
  OPTIONAL { ?person wdt:P18 ?image. }
  OPTIONAL {
    ?person wdt:P512 ?education.
    ?education rdfs:label ?educationLabel . FILTER(LANG(?educationLabel) = "ja")
  }
  OPTIONAL {
    ?person wdt:P106 ?occupation.
    ?occupation rdfs:label ?occupationLabel . FILTER(LANG(?occupationLabel) = "ja")
  }
  OPTIONAL {
    ?person wdt:P734 ?familyName.
    ?familyName rdfs:label ?familyNameLabel . FILTER(LANG(?familyNameLabel) = "ja")
  }
  OPTIONAL {
    ?person wdt:P735 ?givenName.
    ?givenName rdfs:label ?givenNameLabel . FILTER(LANG(?givenNameLabel) = "ja")
  }
  OPTIONAL { ?person wdt:P2048 ?height. }
  OPTIONAL { ?person wdt:P2067 ?weight. }
  OPTIONAL {
    ?person wdt:P509 ?causeOfDeath.
    ?causeOfDeath rdfs:label ?causeOfDeathLabel . FILTER(LANG(?causeOfDeathLabel) = "ja")
  }
  OPTIONAL {
    ?person wdt:P1196 ?mannerOfDeath.
    ?mannerOfDeath rdfs:label ?mannerOfDeathLabel . FILTER(LANG(?mannerOfDeathLabel) = "ja")
  }
}
GROUP BY ?person ?personLabel ?personLabelEn ?article ?genderLabel ?birthDate ?deathDate ?description ?image ?height ?weight
"""
HEADERS = {
    "User-Agent": "MyBot/1.0 (contact: your_email@example.com)",
    "Accept": "application/sparql-results+json",
}

In [7]:
import os
import json
import time
import requests
import pandas as pd

os.makedirs("data", exist_ok=True)

QLEVER_URL = "https://qlever.cs.uni-freiburg.de/api/wikidata"

JSON_PATH = "data/jawikidata.json"
CSV_PATH = "data/jawikidata.csv"


def fetch_json(max_retries=5):
    """SPARQL標準のJSON結果形式で取得する。action指定なし = デフォルトのJSON。"""
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Sending request to QLever (Attempt {attempt}/{max_retries})...")
            r = requests.post(
                QLEVER_URL,
                data={"query": QUERY},  # action は付けない = SPARQL標準JSON
                headers=HEADERS,
                timeout=300,
            )
            if r.status_code == 200:
                return r.json()
            else:
                print(f"  Status error: {r.status_code}, body[:300]={r.text[:300]!r}")
        except requests.exceptions.RequestException as e:
            print(f"  Network error: {e}")
        except json.JSONDecodeError as e:
            print(f"  JSON decode error: {e}, body[:300]={r.text[:300]!r}")

        time.sleep(5 * attempt)

    raise RuntimeError(f"Failed to fetch data after {max_retries} retries.")


def json_to_dataframe(data: dict) -> pd.DataFrame:
    """SPARQL JSON results (head.vars / results.bindings) を DataFrame化する。"""
    cols = data["head"]["vars"]
    bindings = data["results"]["bindings"]

    rows = []
    for b in bindings:
        row = {col: b.get(col, {}).get("value") for col in cols}
        rows.append(row)

    return pd.DataFrame(rows, columns=cols)


def main():
    start_time = time.time()

    # 1. JSON取得
    data = fetch_json()
    n_bindings = len(data.get("results", {}).get("bindings", []))
    print(f"Fetched {n_bindings} rows from QLever.")

    # 2. JSON保存（生データをそのまま残す。デバッグ・再変換用）
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Saved raw JSON -> {JSON_PATH}")

    # 3. CSV化して保存
    df = json_to_dataframe(data)
    df.to_csv(CSV_PATH, index=False, encoding="utf-8")
    print(f"Saved CSV -> {CSV_PATH}")

    elapsed = time.time() - start_time
    print(f"\nDone. Rows: {len(df)}, Columns: {list(df.columns)}")
    print(f"Total time spent: {int(elapsed)}s")


if __name__ == "__main__":
    main()

Sending request to QLever (Attempt 1/5)...
Fetched 236793 rows from QLever.
Saved raw JSON -> data/jawikidata.json
Saved CSV -> data/jawikidata.csv

Done. Rows: 236793, Columns: ['person', 'personLabel', 'personLabelEn', 'article', 'genderLabel', 'birthDate', 'deathDate', 'description', 'image', 'educations', 'occupations', 'familyNames', 'givenNames', 'height', 'weight', 'causesOfDeath', 'mannersOfDeath']
Total time spent: 55s


In [8]:
import pandas as pd

wikidata_raw = pd.read_csv("data/jawikidata.csv", encoding="utf-8")

rename_map = {
    "person":        "qid_url",
    "personLabel":   "name_ja",
    "personLabelEn": "name_en",
    "article":       "wiki_url",
    "genderLabel":   "gender",
    "birthDate":     "birth_date",
    "deathDate":     "death_date",
    "description":   "description",
    "image":         "image_url",
    "educations":    "educations",
    "occupations":   "occupations",
    "familyNames":   "family_name",
    "givenNames":    "given_name",
    "height":        "height_cm",
    "weight":        "weight_kg",
    "causesOfDeath":  "death_causes",
    "mannersOfDeath": "death_manners",
}

wikidata_raw = wikidata_raw.rename(columns=rename_map)

In [ ]:
import romkan2
import unicodedata
import numpy as np

def swap_name_order(name):
    if not isinstance(name, str):
        return name
    parts = name.split()
    if len(parts) == 2:
        # "Given Family" -> "Family Given"
        return f"{parts[1]} {parts[0]}"
    elif len(parts) != 2:
        return ""
    return name  # single-word names, leave as is

MACRON_MAP = {
    'ā': 'aa',
    'ī': 'ii',
    'ū': 'uu',
    'ē': 'ee',
    'ō': 'ou',
    'Ā': 'Aa',
    'Ī': 'Ii',
    'Ū': 'Uu',
    'Ē': 'Ee',
    'Ō': 'Ou',
}

def to_hiragana_safe(name):
    if not isinstance(name, str):
        return name
    for macron, expanded in MACRON_MAP.items():
        name = name.replace(macron, expanded)
    nfkd_form = unicodedata.normalize('NFKD', name)
    name = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    return romkan2.to_hiragana(name)

wikidata = wikidata_raw.dropna(subset=["name_ja"]).copy()
wikidata["hiragana"] = wikidata["name_en"].apply(swap_name_order)
wikidata["hiragana"] = wikidata["hiragana"].apply(to_hiragana_safe)

# only hiragana from start to end
is_pure_hiragana = wikidata["hiragana"].str.match(r'^[\u3040-\u309F\s]+$', na=False)
wikidata["hiragana"] = np.where(is_pure_hiragana, wikidata["hiragana"], "")


from namedivider import GBDTNameDivider

name_divider = GBDTNameDivider()
def split_name(x):
    if pd.isna(x) or len(x) <= 1:
        return x
    result = name_divider.divide_name(x)
    return f"{result.family} {result.given}"

wikidata["name_ja"] = wikidata["name_ja"].map(split_name)


wikidata = wikidata[
    [
        "name_ja",
        "name_en",
        "hiragana",
        "gender",
        "description",
        "educations",
        "occupations",
        "birth_date",
        "death_causes",
        "death_manners",
        "death_date",
        "family_name",
        "given_name",
        "weight_kg",
        "height_cm",
        "qid_url",
        "wiki_url",
        "image_url",
    ]
]


Download FamilyNameRepository from GitHub...
Download GBDT Model from GitHub...


In [ ]:
filtered = wikidata.copy()
filtered = filtered.drop(columns=["family_name", "given_name"])
filtered = filtered.dropna(subset=["name_ja","name_en", "gender"], how="any")
# filtered = filtered[filtered["gender"].isin(["男性", "女性"])].reset_index(drop=True)

import re

def is_japanese_only(text):
    """does text only have 漢字・ひらがな・カタカナ（+ 々・ー・space）"""
    if not isinstance(text, str) or text == "":
        return False
    pattern = r'^[\u4E00-\u9FFF\u3040-\u309F\u30A0-\u30FF\u3005\u30FC\s]+$'
    return bool(re.fullmatch(pattern, text))

def is_kanji_only(text):
    """漢字（+ 々・ー）で構成されているか判定"""
    if not isinstance(text, str) or text == "":
        return False
    pattern = r'^[\u4E00-\u9FFF\u3005\u30FC]+$'
    return bool(re.fullmatch(pattern, text))


filtered["name_ja"] = (
    filtered["name_ja"]
    .str.replace(r'[\(（].*$', '', regex=True)
    .str.strip()
)
filtered = filtered[filtered["name_ja"].apply(is_japanese_only)] # 1k+ drop non japanese

# keep the old people with "no" in the reading by removing it
mask = (
    filtered["name_en"].str.contains(" no ", case=True, na=False)
    & filtered["name_ja"].apply(is_kanji_only)
)
filtered.loc[mask, "name_en"] = filtered.loc[mask, "name_en"].str.replace(" no ", "", regex=False)

filtered["hiragana"] = filtered["name_en"].apply(to_hiragana_safe)
filtered["hiragana"] = filtered["hiragana"].apply(swap_name_order)
filtered = filtered[filtered["hiragana"].apply(is_japanese_only)] # 3k drop
filtered = filtered[filtered["hiragana"] != ""] # 4.6k drop split != 2

In [ ]:
# Grab 4 characters, force text like 'http' to become NaN, and save as nullable integers
filtered["birth_year"] = pd.to_numeric(filtered["birth_date"].str[:4], errors="coerce").astype("Int64")
filtered["death_year"] = pd.to_numeric(filtered["death_date"].str[:4], errors="coerce").astype("Int64")
filtered["age"] = filtered["death_year"] - filtered["birth_year"]
filtered["age"] = filtered["age"].where(filtered["age"] > 0, None)

In [ ]:
1-filtered.isna().sum()/filtered.shape[0]

In [ ]:
filtered["height_cm"] = np.where(
    filtered["height_cm"] <= 5, 
    filtered["height_cm"] * 100, 
    filtered["height_cm"]
)
filtered["bmi"] = filtered["weight_kg"] / ((filtered["height_cm"] / 100) ** 2)

In [ ]:
filtered["bmi"][filtered["bmi"]<100]

In [ ]:
filtered[(filtered["height_cm"] >= 5) & (filtered["height_cm"] <= 120)][["name_ja","gender","occupation","weight_kg","height_cm","age","bmi"]]

In [ ]:

filtered = filtered[
    [
        "name_ja",
        "name_en",
        "hiragana",
        "gender",
        "description",
        "educations",
        "occupations",
        "death_causes",
        "death_manners",
        "birth_year",
        "death_year",
        "age",
        "weight_kg",
        "height_cm",
        "qid_url",
        "wiki_url",
        "image_url",
    ]
]

filtered = filtered.reset_index(drop=True)

In [ ]:
wikidata.to_csv("jpersondata_raw.csv")
filtered.to_csv("jpersondata.csv")